# Python 知识

# 代码分析

## 类继承关系

```mermaid
classDiagram
    class type {
        <<metaclass>>
    }
    class MetaFields {
        <<metaclass>>
    }
    class RecordsWithFields
    class MetaStatsBuilderMixin {
        <<metaclass>>
    }
    class MetaPlotsBuilderMixin {
        <<metaclass>>
    }
    class StatsBuilderMixin
    class PlotsBuilderMixin
    class MetaRecords {
        <<metaclass>>
    }

    class Wrapping
    class Records

    
    %% 继承关系
    type <|-- MetaFields : metaclass

    MetaFields <|-- RecordsWithFields : metaclass

    MetaStatsBuilderMixin <|-- StatsBuilderMixin : metaclass
    MetaPlotsBuilderMixin <|-- PlotsBuilderMixin : metaclass

    MetaStatsBuilderMixin <|-- MetaRecords : metaclass
    MetaPlotsBuilderMixin <|-- MetaRecords : metaclass
    MetaFields <|-- MetaRecords : metaclass

    Wrapping <|-- Records
    StatsBuilderMixin <|-- Records
    PlotsBuilderMixin <|-- Records
    RecordsWithFields <|-- Records
    MetaRecords <|-- Records : metaclass
```

## `class MetaFields(type)` 和 `class RecordsWithFields(metaclass=MetaFields)`

```python
class MetaFields(type):
    @property
    def field_config(cls) -> Config:
        return cls._field_config
```

```python
class RecordsWithFields(metaclass=MetaFields):
    @property
    def field_config(self) -> Config:
        return self._field_config
```

## `class Records(Wrapping, StatsBuilderMixin, PlotsBuilderMixin, RecordsWithFields, metaclass=MetaRecords)`
`self.wrapper` 是包装器，记录 `index` 和 `columns`。

`self._records_arr` 是结构化 np.ndarray，是真正要存储的数据。

`self._col_mapper` 记录了 `self._records_arr` 的每一项数据对应 `columns` 中的哪一列。

### _field_config
一个类变量，定义 `Records` 类的基本字段结构。

```python
_field_config: tp.ClassVar[Config] = Config(
    dict(
        # dtype字段：定义记录数组的数据类型结构
        # 默认为None，子类可以重写以定义具体的数据类型
        dtype=None,
        
        # settings字段：定义各个字段的配置信息
        settings=dict(
            # id字段配置：记录的唯一标识符
            id=dict(
                name='id',        # 字段在数组中的实际名称
                title='Id'        # 字段的显示标题
            ),
            
            # col字段配置：记录所属的列索引
            col=dict(
                name='col',           # 字段在数组中的实际名称
                title='Column',       # 字段的显示标题
                mapping='columns'     # 映射到ArrayWrapper的columns属性
            ),
            
            # idx字段配置：记录的时间索引
            idx=dict(
                name='idx',           # 字段在数组中的实际名称
                title='Timestamp',    # 字段的显示标题
                mapping='index'       # 映射到ArrayWrapper的index属性
            )
        )
    ),
    readonly=True,        # 配置为只读，防止意外修改
    as_attrs=False       # 不将配置项作为属性访问
)
```

### `__init__`

```python
def __init__(self,
                wrapper: ArrayWrapper,
                records_arr: tp.RecordArray,
                col_mapper: tp.Optional[ColumnMapper] = None,
                **kwargs) -> None:
    Wrapping.__init__(
        self,
        wrapper,
        records_arr=records_arr,
        col_mapper=col_mapper,
        **kwargs
    )
    StatsBuilderMixin.__init__(self)

    # Check fields
    records_arr = np.asarray(records_arr)
    checks.assert_not_none(records_arr.dtype.fields)

    # 从字段配置 self.field_config[settings] 中提取所有字段名称
    field_names = {
        dct.get('name', field_name)
        for field_name, dct in self.field_config.get('settings', {}).items()
    }

    # 如果 self.field_config[dtype].names 中存在不属于 records_arr.dtype.names 以及 field_names 的项，抛出异常
    dtype = self.field_config.get('dtype', None)
    if dtype is not None:
        for field in dtype.names:
            if field not in records_arr.dtype.names:
                if field not in field_names:
                    raise TypeError(f"Field '{field}' from {dtype} cannot be found in records or config")

    self._records_arr = records_arr
    if col_mapper is None:
        col_mapper = ColumnMapper(wrapper, self.col_arr)
    self._col_mapper = col_mapper
```

## `Records` 使用例子

### 创建 ArrayWrapper (wrapper 参数)

In [ ]:
import numpy as np
import pandas as pd
import vectorbt as vbt

# ================================================================
# 第一步：创建 ArrayWrapper (wrapper 参数)
# ================================================================

print("=" * 60)
print("第一步：创建 ArrayWrapper")
print("=" * 60)

# ArrayWrapper 是 Records 的核心元数据管理器
# 它定义了：
# 1. 时间索引（index）- 什么时间点
# 2. 列名（columns）- 哪些资产/策略
# 3. 数组形状（shape）- 数据的维度
# 4. 分组信息（可选）- 如何分组

# 创建时间索引：交易日期
time_index = pd.date_range('2023-01-01', periods=5, freq='D')
print(f"时间索引: {time_index}")

# 创建列名：三只股票
stock_columns = ['AAPL', 'GOOGL', 'MSFT']
print(f"股票列名: {stock_columns}")

# 创建数组包装器
wrapper = vbt.ArrayWrapper(
    index=time_index,        # 时间索引
    columns=stock_columns,   # 列名
    ndim=2,                  # 二维数组
    freq='D'                 # 时间频率
)

print(f"ArrayWrapper 形状: {wrapper.shape}")  # (5, 3) - 5个时间点，3只股票
print(f"ArrayWrapper 列数: {wrapper.shape[1]}")
print(f"ArrayWrapper 时间点数: {wrapper.shape[0]}")

### 创建结构化数组 (records_arr 参数)

In [ ]:
print("\n" + "=" * 60)
print("第二步：创建结构化数组")
print("=" * 60)

# 结构化数组是 Records 的核心数据存储
# 它必须包含特定的字段：
# 1. 'id' - 记录的唯一标识符
# 2. 'col' - 记录属于哪一列（股票）
# 3. 'idx' - 记录在哪个时间点
# 4. 其他自定义字段 - 如价格、成交量等

# 定义记录的数据结构
trade_dtype = np.dtype([
    ('id', np.int64),        # 交易ID
    ('col', np.int64),       # 列索引（0=AAPL, 1=GOOGL, 2=MSFT）
    ('idx', np.int64),       # 时间索引（0-4对应5个交易日）
    ('price', np.float64),   # 交易价格
    ('volume', np.int64),    # 交易量
    ('side', np.int8),       # 交易方向（0=买入, 1=卖出）
    ('pnl', np.float64)      # 盈亏
])

# 创建交易记录数据
# 模拟一些交易记录：不同时间点、不同股票的交易
records_data = np.array([
    # (id, col, idx, price, volume, side, pnl)
    (0,  0,   0,   150.0,  1000,  0,   0.0),      # AAPL 第0天 买入
    (1,  0,   1,   152.0,  1000,  1,   2000.0),   # AAPL 第1天 卖出，盈利2000
    (2,  1,   0,   2800.0, 100,   0,   0.0),      # GOOGL 第0天 买入
    (3,  1,   2,   2850.0, 100,   1,   5000.0),   # GOOGL 第2天 卖出，盈利5000
    (4,  2,   1,   380.0,  500,   0,   0.0),      # MSFT 第1天 买入
    (5,  2,   3,   385.0,  500,   1,   2500.0),   # MSFT 第3天 卖出，盈利2500
    (6,  0,   3,   148.0,  2000,  0,   0.0),      # AAPL 第3天 再次买入
    (7,  0,   4,   150.0,  2000,  1,   4000.0),   # AAPL 第4天 卖出，盈利4000
    (8,  1,   4,   2900.0, 200,   0,   0.0),      # GOOGL 第4天 买入
    (9,  2,   4,   390.0,  800,   0,   0.0),      # MSFT 第4天 买入
], dtype=trade_dtype)

print(f"交易记录数量: {len(records_data)}")
print(f"记录数据类型: {records_data.dtype}")
print(f"字段名称: {records_data.dtype.names}")

# 查看部分记录
print("\n交易记录:")
for i in range(len(records_data)):
    record = records_data[i]
    stock_name = stock_columns[record['col']]
    date = time_index[record['idx']]
    side_str = '买入' if record['side'] == 0 else '卖出'
    print(f"ID{record['id']}: {stock_name} {date.strftime('%Y-%m-%d')} {side_str} "
          f"价格${record['price']:.2f} 数量{record['volume']} 盈亏${record['pnl']:.2f}")

### 创建 ColumnMapper (col_mapper 参数)

In [ ]:
from vectorbt.records.col_mapper import ColumnMapper

print("\n" + "=" * 60)
print("第三步：创建 ColumnMapper (可选)")
print("=" * 60)

# ColumnMapper 是用于优化列操作的组件
# 它根据数据的排序状态选择最优的索引策略：
# 1. 如果数据按列排序 -> 使用 col_range (更快)
# 2. 如果数据未排序 -> 使用 col_map (更灵活)

# 提取列数组
col_arr = records_data['col']
print(f"列数组: {col_arr}")

# 创建列映射器
col_mapper = ColumnMapper(wrapper, col_arr)

print(f"数据是否按列排序: {col_mapper.is_sorted()}")
print(f"列映射器类型: {type(col_mapper)}")

# 查看列映射器的内部信息
if col_mapper.is_sorted():
    print("使用列范围索引 (col_range) - 适合排序数据")
    print(f"列范围: {col_mapper.col_range}")
else:
    print("使用列映射索引 (col_map) - 适合未排序数据")
    print(f"列映射: {col_mapper.col_map}")

### 创建 Records 对象

In [ ]:
print("\n" + "=" * 60)
print("第四步：创建 Records 对象")
print("=" * 60)

# 方法1：使用所有三个参数
records_full = vbt.Records(
    wrapper=wrapper,           # ArrayWrapper：提供元数据
    records_arr=records_data,  # 结构化数组：存储记录数据
    col_mapper=col_mapper      # ColumnMapper：优化列操作（可选）
)

print("使用完整参数创建的 Records 对象:")
print(f"记录数量: {len(records_full)}")
print(f"包装器: {records_full.wrapper}")
print(f"列映射器: {records_full.col_mapper}")

# 方法2：不提供 col_mapper（Records 会自动创建）
records_auto = vbt.Records(
    wrapper=wrapper,
    records_arr=records_data
    # col_mapper 会自动创建
)

print("\n使用自动创建 col_mapper 的 Records 对象:")
print(f"记录数量: {len(records_auto)}")
print(f"自动创建的列映射器: {records_auto.col_mapper}")

### Records 的基本功能 

In [ ]:
print("\n" + "=" * 60)
print("第五步：Records 基本功能展示")
print("=" * 60)

# 5.1 基本信息
print("5.1 基本信息:")
print(f"记录总数: {len(records_full)}")
print(f"字段配置: {records_full.field_config}")

In [ ]:
# 5.2 字段访问
print("\n5.2 字段访问:")
print(f"ID 数组: {records_full.id_arr}")
print(f"列数组: {records_full.col_arr}")
print(f"时间索引数组: {records_full.idx_arr}")
print(f"价格数组: {records_full.get_field_arr('price')}")

In [ ]:
# 5.3 原始记录 vs 可读记录
print("\n5.3 原始记录 vs 可读记录:")
print("原始记录 (raw records):")
print(records_full.records)

print("\n可读记录 (readable records):")
print(records_full.records_readable)

In [ ]:
# 5.4 字段映射
print("\n5.4 字段映射:")
price_mapped = records_full.map_field('price')
volume_mapped = records_full.map_field('volume')
pnl_mapped = records_full.map_field('pnl')

print(f"价格映射数组: {price_mapped}")
print(f"成交量映射数组: {volume_mapped}")
print(f"盈亏映射数组: {pnl_mapped}")

In [ ]:
# 5.5 统计分析
print("\n5.5 统计分析:")
print("价格统计:")
print(price_mapped.describe())

print("\n总盈亏:")
print(pnl_mapped.sum())

print("\n各股票盈亏:")
print(pnl_mapped.sum(group_by=False))

In [ ]:
# 5.6 数据过滤
print("\n5.6 数据过滤:")
# 过滤出买入交易
buy_mask = records_full.get_field_arr('side') == 0
buy_records = records_full.apply_mask(buy_mask)
print(f"买入交易数量: {len(buy_records)}")

# 过滤出高价格交易
high_price_mask = records_full.get_field_arr('price') > 200
high_price_records = records_full.apply_mask(high_price_mask)
print(f"高价格交易数量: {len(high_price_records)}")

In [ ]:
# 5.7 分组操作
print("\n5.7 分组操作:")
# 按行业分组：科技股 vs 其他
group_by_industry = ['Tech', 'Tech', 'Other']
industry_pnl = pnl_mapped.sum(group_by=group_by_industry)
print(f"按行业分组的盈亏: {industry_pnl}")